# ODI to Databricks Migration: TRG_DEP Load

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook performs a direct insert of data from `HR.DEPARTMENTS` into `HR.TRG_DEP`.
The original ODI script contained a single `INSERT` statement for a full load scenario.

In [ ]:
import dbutils

dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "2. Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "1000", "3. ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "12345", "4. ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no;

In [ ]:
display(spark.sql("""
  SELECT
    v_etl_job_type.etl_job_type,
    v_datasource_num_id.datasource_num_id,
    v_etl_proc_wid.etl_proc_wid,
    v_odi_sess_no.odi_sess_no
  FROM v_etl_job_type, v_datasource_num_id, v_etl_proc_wid, v_odi_sess_no
"""))

## Target Table Load

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Create Target Table
-- SCEN_TASK_NO in {20}: (Empty task in original ODI, no direct conversion needed)

CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
  department_id   BIGINT,
  department_name STRING,
  manager_id      BIGINT,
  location_id     BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO in {30}: Insert into Target Table

INSERT INTO workspace.hr.trg_dep
(
  department_id,
  department_name,
  manager_id,
  location_id
)
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

## Validation

In [ ]:
%sql
SELECT
  COUNT(*) AS total_records
FROM
  workspace.hr.trg_dep;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema and table names have been converted to lowercase and prefixed with `workspace.` as per standards.
    *   `HR.TRG_DEP` -> `workspace.hr.trg_dep`
    *   `HR.DEPARTMENTS` -> `workspace.hr.departments`
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` Oracle hint has been removed as it is not applicable to Databricks Delta Lake.
3.  **Task {10} & {30}:** Corresponds to target table creation and the primary insert statement.
4.  **Task {20}:** Was an empty task in the original ODI and has been noted as such in a comment within cell 6.
5.  **Data Types:** Inferred data types for `workspace.hr.trg_dep` are `BIGINT` for ID columns and `STRING` for name columns, based on typical Oracle HR schema definitions.
6.  **Full Load Assumption:** This conversion assumes a full load/append strategy as no `DELETE` or `TRUNCATE` statements were present in the source ODI SQL. If `TRG_DEP` is intended for a full refresh, a `TRUNCATE TABLE workspace.hr.trg_dep;` statement should be added before the `INSERT`.